# Go / no-go: does alarm-aware compression preserve a VLA's failure monitor better than action-aware compression?

**Model:** SmolVLA (`HuggingFaceVLA/smolvla_libero`) · **Benchmark:** LIBERO-Spatial · **GPU:** Kaggle 2× T4 · **Code:** `vla-assure`

Same experiment and decision table as the Colab notebook; only the runtime plumbing differs.

**Question.** If we quantize SmolVLA's action expert to 4 bits: (1) does the inherited Mahalanobis monitor's false-alarm rate move, and (2) with ~15 % of the expert kept in full precision, does choosing those blocks by their effect on the **alarm statistic** keep the monitor closer to fp16 than choosing them by **action** change or at random?

### One-time setup (5 min)
1. Kaggle account with **phone verification** (needed for GPU and internet).
2. **Datasets → New dataset**, upload `vla-assure.zip` (the latest one I sent), name it `vla-assure`.
3. New notebook → **File → Import notebook** → this file. Right panel: **Accelerator = GPU T4 x2**, **Internet = On**, **Add input** → your `vla-assure` dataset.

### Running it
Run it as a background job: **Save Version → Save & Run All (Commit)**. It runs up to 12 h without the browser open, and everything in `/kaggle/working` is saved as the version's output. The notebook stops itself cleanly at `TIME_BUDGET_H` (11 h) if it is not done.

**To continue after a stopped run:** *Add input → Notebook output →* this notebook's latest version, then Save & Run All again. Setup reinstalls (~10 min), finished episodes are skipped, and it carries on. Expect **2 runs** in total (~25–35 GPU-h of the 30 h weekly quota with both T4s counted; check the measured time in step 1g).

| Step | What | Episodes | Rough 2×T4 time* |
|---|---|---|---|
| 1 | Setup, install, smoke test | 2 | 15 min |
| 2 | Stage 0-lite: fp16 nominal sanity | 100 | 0.5–1 h |
| 3 | Pick the camera condition (60–85 % success) | 100 | 0.5–1 h |
| 4 | Reference fp16 run, 50 slots/task | 500 | 2–4 h |
| 5 | Block sensitivity | ≤ 96 captured | 1–2 h |
| 6 | Four compressed variants on the 175 test slots | 700 | 3–5 h |
| 7 | Evaluation and decision table | — | 5 min |

\*Re-estimated from the measured seconds/episode in step 1g. When it finishes, download `vla-gonogo/gonogo_summary.zip` from the version's **Output** tab and send it to me.

## 1. Setup

In [ ]:
# 1a. GPUs, time guard, work folder; resume from a previous version's output if one is attached as input.
import os, sys, time, json, glob, shutil, pathlib, subprocess
T_START = time.time()
TIME_BUDGET_H = 11.0          # stop cleanly before Kaggle's 12 h limit so the output is saved
def hours_left():
    return TIME_BUDGET_H - (time.time() - T_START) / 3600

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
assert NGPU >= 1, "No GPU: right panel > Accelerator > GPU T4 x2"
print("GPUs:", NGPU, "| kernel Python", sys.version.split()[0])

WORK = pathlib.Path("/kaggle/working/vla-gonogo"); WORK.mkdir(parents=True, exist_ok=True)
# Kaggle mounts inputs at different depths (e.g. /kaggle/input/notebooks/<user>/<slug>/vla-gonogo)
prev = sorted({p for p in glob.glob("/kaggle/input/**/vla-gonogo", recursive=True) if pathlib.Path(p).is_dir()})
for p in prev:
    if pathlib.Path(p).resolve() != WORK.resolve():
        print("Resuming from previous output:", p)
        shutil.copytree(p, WORK, dirs_exist_ok=True, symlinks=False)
n_done = len(list(WORK.rglob("*.npz")))
print("Work folder:", WORK, "|", n_done, "episodes already on disk")
PERSISTENT = True   # /kaggle/working is saved as the version's output

In [ ]:
# 1b. Code: the LATEST vla-assure.zip, uploaded as a Kaggle dataset and attached as input.
import zipfile
CODE = pathlib.Path("/root/vla-assure")

def code_ok(root):
    f = root / "src/assure/adapters/lerobot_adapter.py"
    return f.exists() and "_chunk_method" in f.read_text() and "_require_lerobot_06" in f.read_text()

if CODE.exists() and not code_ok(CODE):
    shutil.rmtree(CODE)
if not CODE.exists():
    zips = sorted(glob.glob("/kaggle/input/**/vla-assure*.zip", recursive=True))
    # Kaggle sometimes unpacks an uploaded zip into a folder instead
    trees = sorted({str(pathlib.Path(p).parents[3]) for p in
                    glob.glob("/kaggle/input/**/src/assure/adapters/lerobot_adapter.py", recursive=True)})
    if zips:
        print("Using", zips[0])
        tmp = pathlib.Path("/root/_unzip"); shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(zips[0]) as f:
            f.extractall(tmp)
        root = next((p.parents[3] for p in tmp.rglob("src/assure/adapters/lerobot_adapter.py")), None)
        assert root is not None, f"{zips[0]} does not contain the vla-assure code"
        shutil.move(str(root), CODE)
    elif trees:
        print("Using unpacked dataset", trees[0])
        shutil.copytree(trees[0], CODE, symlinks=False)
    else:
        raise FileNotFoundError("vla-assure not found under /kaggle/input: right panel > Add input > your vla-assure dataset")
assert code_ok(CODE), "This zip is outdated: upload the vla-assure.zip from the latest message as a new dataset version"
for d in ("rollouts", "results"):
    (WORK / d).mkdir(exist_ok=True)
    tgt = CODE / d
    if tgt.is_symlink():
        tgt.unlink()
    elif tgt.exists():
        shutil.rmtree(tgt)
    tgt.symlink_to(WORK / d)
os.chdir(CODE); print("Code OK at", os.getcwd())

In [ ]:
# 1c. Python 3.12 venv for LeRobot 0.6.1 (needs Python >= 3.12; Kaggle's kernel env is left untouched).
#     egl_probe / hf-egl-probe (via hf-libero -> robomimic) are C++ packages that fail to build with CMake 4;
#     LeRobot's LIBERO env never imports them, so tiny pure-Python stand-ins are installed first.
VENV = pathlib.Path("/root/py312"); PY = str(VENV / "bin/python")
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       env=dict(os.environ, CMAKE_POLICY_VERSION_MINIMUM="3.5"))
    if r.returncode != 0 or not check:
        print(r.stdout[-1500:], r.stderr[-3000:])
    if check:
        assert r.returncode == 0, f"failed: {cmd}"
    return r
if not pathlib.Path(PY).exists():
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv --seed --python 3.12 {VENV}")
stub = pathlib.Path("/root/_stubs")
(stub / "egl_probe/egl_probe").mkdir(parents=True, exist_ok=True); (stub / "hf_egl_probe").mkdir(parents=True, exist_ok=True)
(stub / "egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "egl_probe"\nversion = "1.0.2"\n[tool.setuptools]\npackages = ["egl_probe"]\n')
(stub / "egl_probe/egl_probe/__init__.py").write_text("def get_available_devices():\n    return [0]\n")
(stub / "hf_egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "hf-egl-probe"\nversion = "1.0.2"\n[tool.setuptools]\npy-modules = []\n')
sh(f"{PY} -m pip install -q {stub / 'egl_probe'} {stub / 'hf_egl_probe'}")
t = time.time()
sh(f'{PY} -m pip install -q "lerobot[smolvla,libero]==0.6.1" pandas scipy tabulate pyyaml')
print(f"installed in {time.time() - t:.0f} s:",
      sh(f"{PY} -c \"import importlib.metadata as m, torch; print('lerobot', m.version('lerobot'), '| torch', "
         f"torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.device_count())\"").stdout)

In [ ]:
# 1d. Headless rendering. Try NVIDIA EGL first; if MuJoCo cannot render, install Mesa EGL (software) and retry;
#     last resort OSMesa. The chosen backend is used by every later step.
probe = ("import mujoco; m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><light pos=\"0 0 1\"/>"
         "<geom type=\"sphere\" size=\".1\"/></worldbody></mujoco>'); r = mujoco.Renderer(m, 64, 64); "
         "d = mujoco.MjData(m); mujoco.mj_forward(m, d); r.update_scene(d); print('render ok', r.render().shape)")
def try_gl(gl):
    r = subprocess.run([PY, "-c", probe], capture_output=True, text=True,
                       env=dict(os.environ, MUJOCO_GL=gl, PYOPENGL_PLATFORM=gl))
    return r.returncode == 0, (r.stdout + r.stderr)[-400:]
GL = None
for attempt in ("egl", "egl+mesa", "osmesa"):
    if attempt == "egl+mesa":
        sh("apt-get -qq update && apt-get -qq install -y libegl1 libegl-mesa0 libgl1 libgl1-mesa-dri >/dev/null", check=False)
    if attempt == "osmesa":
        sh("apt-get -qq install -y libosmesa6 >/dev/null", check=False)
    ok, log = try_gl(attempt.split("+")[0])
    print(attempt, "->", "OK" if ok else log)
    if ok:
        GL = attempt.split("+")[0]; break
assert GL, "no rendering backend works"
if attempt == "egl+mesa":
    print("NOTE: software EGL (Mesa). Rendering is CPU-bound; expect slower episodes than on Colab.")
os.environ["MUJOCO_GL"] = GL; os.environ["PYOPENGL_PLATFORM"] = GL
os.environ["MPLBACKEND"] = "Agg"   # the kernel's inline backend is not valid inside the venv's matplotlib
# LIBERO's first import asks interactive questions and downloads assets: answer once, non-interactively.
r = subprocess.run([PY, "-c", "import libero.libero as L; print('assets:', L.get_libero_path('assets'))"],
                   input="n\n", capture_output=True, text=True, env=dict(os.environ))
print(r.stdout[-600:], r.stderr[-600:])

Configuration and helpers (same settings as the Colab notebook).

In [ ]:
# 1e. Experiment configuration for this gate (written into the repo copy on this machine only)
import yaml, json
CFG = CODE / "configs"
exp = yaml.safe_load((CFG / "experiment.yaml").read_text())
models = yaml.safe_load((CFG / "models.yaml").read_text())

MODEL, SUITE = "smolvla", "libero_spatial"
R_REF = 50              # reference slots per task (500 slots: 125 train / 100 calib / 100 repair / 175 test)
BUDGET_PCT = 15         # % of the ACTION EXPERT's linear weights kept in full precision (~5 of 32 blocks)
N_ACTION_STEPS = 10     # 10 actions per model call (~5x faster than the checkpoint's 1); same for every variant

exp["quant_scope"] = {"include": ["model.vlm_with_expert.lm_expert.*"],
                      "block_regex": r"lm_expert\.layers\.(\d+)$"}
exp["monitors"] = ["mahalanobis#A", "mahalanobis"]
exp["reference_episodes_per_task"] = R_REF
models["models"][MODEL]["checkpoint"] = "HuggingFaceVLA/smolvla_libero"
models["models"][MODEL]["n_action_steps"] = N_ACTION_STEPS
(CFG / "experiment.yaml").write_text(yaml.safe_dump(exp, sort_keys=False))
(CFG / "models.yaml").write_text(yaml.safe_dump(models, sort_keys=False))
CONDITION = (WORK / "condition.txt").read_text().strip() if (WORK / "condition.txt").exists() else None
KEY = f"{SUITE}+{CONDITION}" if CONDITION else None
print("condition from an earlier session:", CONDITION)
VARIANTS = ["w4", f"w4p-margin_drift-f{BUDGET_PCT}", f"w4p-action_drift-f{BUDGET_PCT}", f"w4p-random-f{BUDGET_PCT}"]
print("variants:", VARIANTS)
print("taps:", models["models"][MODEL]["tap"])

def run(cmd, tail=3000):
    # Run a pipeline script, stream nothing, print the end of the log, raise on failure.
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    p = subprocess.run([PY] + cmd, capture_output=True, text=True, env=env, input="n\n")
    out = (p.stdout + p.stderr).replace("\r", "\n")
    out = "\n".join(l for l in out.splitlines() if "Running rollout" not in l and "it/s" not in l)
    print(out[-tail:])
    if p.returncode != 0:
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(cmd)}")


class OutOfTime(Exception):
    pass

def _clean_partial(since):
    # remove episode files that a killed shard may have left half-written
    import numpy as np
    for f in (CODE / "rollouts").rglob("*.npz"):
        if f.stat().st_mtime >= since - 120:
            try:
                np.load(f)["success"]
            except Exception:
                print("removing partial file", f); f.unlink()

def collect(args, tail=1500):
    # 01_collect.py sharded by task over the GPUs, then one merge pass (skips finished episodes) that writes the
    # complete run.json. Stops cleanly when TIME_BUDGET_H is reached; rerun (next version) to resume.
    if hours_left() < 0.25:
        raise OutOfTime("time budget reached: Save Version now; attach this output and run again to resume")
    args = list(args) + ([] if "--skip_existing" in args else ["--skip_existing"])
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    if NGPU > 1 and "--tasks" not in args:
        logs = [open(f"/root/shard{g}.log", "w") for g in range(NGPU)]
        t0 = time.time()
        procs = [subprocess.Popen([PY, "scripts/01_collect.py"] + args +
                                  ["--tasks", ",".join(str(t) for t in range(10) if t % NGPU == g)],
                                  # all shards on GPU 0 (the setup that works with robosuite's EGL checks); SmolVLA is small and the
                                  # simulator is CPU-bound, so two processes on one T4 still give ~2x
                                  env=dict(env, CUDA_VISIBLE_DEVICES="0"), stdout=logs[g], stderr=subprocess.STDOUT,
                                  stdin=subprocess.PIPE, text=True) for g in range(NGPU)]
        for p in procs:
            p.stdin.write("n\n"); p.stdin.close()
        # poll: stop all shards at once if one fails or the time budget runs out
        bad = []
        while any(p.poll() is None for p in procs):
            bad = [g for g, p in enumerate(procs) if p.poll() not in (None, 0)]
            late = hours_left() * 3600 < 600
            if bad or late:
                for p in procs:
                    if p.poll() is None:
                        p.kill()
                for p in procs:
                    p.wait()
                _clean_partial(t0)
                if late and not bad:
                    raise OutOfTime("time budget reached during collection: Save Version; attach this output and run again")
                break
            time.sleep(10)
        for f in logs:
            f.close()
        for g, p in sorted(enumerate(procs), key=lambda x: x[0] not in bad):   # the shard that failed first
            if p.returncode != 0:
                print(open(f"/root/shard{g}.log").read()[-3000:])
                raise RuntimeError(f"shard {g} failed")
    run(["scripts/01_collect.py"] + args, tail=tail)   # merge pass (or the single-GPU run)


In [ ]:
# 1f. Smoke test: model loads, hooks fire, camera shift applied, 2 episodes run. Measures seconds per episode.
run(["scripts/01_collect.py", "--model", MODEL, "--variant", "fp16", "--suite", SUITE, "--condition", "nominal",
     "--tasks", "0", "--episodes", "2", "--out", str(WORK / "rollouts_smoke")])
rj = json.loads((WORK / f"rollouts_smoke/{MODEL}/fp16/{SUITE}/run.json").read_text())
sec = rj["seconds"] / max(rj["episodes"], 1)
print(f"\nsuccess {rj['success_rate']:.2f} | taps fired {rj['tap_fired']} | {sec:.0f} s/episode")
assert all(v > 0 for v in rj["tap_fired"].values()), "feature hooks did not fire: check tap paths in models.yaml"
for step, n in (("Stage 0-lite", 100), ("condition pilot", 100), ("reference", 10 * R_REF), ("variants", 4 * 175)):
    print(f"  {step:16s} {n:5d} episodes ~ {n * sec / 3600:5.1f} h")
print(f"With {NGPU} GPUs in parallel divide by {NGPU}. If reference + variants exceeds ~16 h, set "
      "N_ACTION_STEPS = 10 in 1e and rerun.")

## 2. Stage 0-lite: does fp16 SmolVLA behave as published?
Ten trials per task on nominal LIBERO-Spatial. This only catches a broken setup (wrong preprocessing, wrong checkpoint). The full compatibility check (500 episodes) is part of the paper's Stage 0, not this gate. Fill in the success rate you compare against (model card or the SmolVLA paper).

In [ ]:
PUBLISHED_SPATIAL = None   # e.g. 0.90 -- fill in from the source you compare against
collect(["--model", MODEL, "--variant", "fp16", "--suite", SUITE, "--condition", "nominal",
     "--episodes", "10", "--out", str(WORK / "rollouts_stage0")])
if PUBLISHED_SPATIAL:
    run(["scripts/stage0_check.py", "--model", MODEL, "--key", SUITE, "--published", str(PUBLISHED_SPATIAL),
         "--rollouts", str(WORK / "rollouts_stage0")])
else:
    rj = json.loads((WORK / f"rollouts_stage0/{MODEL}/fp16/{SUITE}/run.json").read_text())
    print("fp16 nominal success:", round(rj["success_rate"], 3), "over", rj["episodes"], "episodes")

## 3. Pick the camera condition
The monitor needs failures and the paired analysis needs successes: target 60–85 % fp16 success. Five trials per task under each seeded camera shift; the condition closest to 72 % inside the window is chosen automatically (override `CONDITION` if you prefer).

In [ ]:
pilot = {}
for cond in ("shift_mild", "shift_strong"):
    collect(["--model", MODEL, "--variant", "fp16", "--suite", SUITE, "--condition", cond,
         "--episodes", "5", "--out", str(WORK / "rollouts_pilot")], tail=400)
    pilot[cond] = json.loads((WORK / f"rollouts_pilot/{MODEL}/fp16/{SUITE}+{cond}/run.json").read_text())["success_rate"]
print("pilot success:", pilot)
inside = {c: p for c, p in pilot.items() if 0.60 <= p <= 0.85}
CONDITION = min(inside or pilot, key=lambda c: abs(pilot[c] - 0.72))
KEY = f"{SUITE}+{CONDITION}"
(WORK / "condition.txt").write_text(CONDITION)
print("CONDITION =", CONDITION, "| KEY =", KEY,
      "" if inside else "\nWARNING: no condition in 60-85 %: edit experiment.yaml conditions (noise/brightness) and rerun")

## 4. Reference run (fp16): 500 pre-registered slots
Every slot has a fixed role (train / calibration / repair / test), seed, initial state and camera draw. The monitor is trained on the training slots only; calibration slots fit the conformal band; test slots are what every variant is compared on. **Resumable** across Kaggle runs.

In [ ]:
collect(["--model", MODEL, "--variant", "fp16", "--suite", SUITE, "--condition", CONDITION,
     "--episodes", str(R_REF)])
rj = json.loads((CODE / f"rollouts/{MODEL}/fp16/{KEY}/run.json").read_text())
n_files = len(list((CODE / f"rollouts/{MODEL}/fp16/{KEY}").glob("*.npz")))
print(f"reference: {n_files}/{10 * R_REF} episodes on disk, success {rj['success_rate']:.3f}")
assert n_files >= 10 * R_REF, "not finished yet: rerun this cell"

## 5. Block sensitivity: alarm-aware and action-aware rankings
Captured reference episodes (outside the slot partition) are replayed with one expert block quantized at a time. The **alarm-aware** score is the change of the monitor's alarm statistic, averaged over successes and failures; the **action-aware** score is the change of the predicted action chunk. Rankings use the selection half of the captured episodes only.

In [ ]:
SENS = CODE / f"results/sensitivity/{MODEL}.json"
if not SENS.exists() and hours_left() < 2:
    raise OutOfTime("less than 2 h left: Save Version; attach this output and run again")
if not SENS.exists():
  run(["scripts/02_sensitivity.py", "--model", MODEL, "--suite", SUITE, "--condition", CONDITION, "--ref_key", KEY,
     "--monitor", "mahalanobis#A", "--select_episodes", "16", "--holdout_episodes", "16",
     "--min_failures", "6", "--min_successes", "6", "--max_captured", "96", "--max_queries", "12", "--rank_boot", "200"],
    tail=2500)
sens = json.loads(SENS.read_text())
import pandas as pd
blk = pd.DataFrame(sens["blocks"])
blk["block"] = blk["name"].str.extract(r"layers\.(\d+)$").astype(int)
blk["type"] = (blk["block"] % 2).map({0: "self-attn", 1: "cross-attn"})
for col in ("margin_drift", "action_drift"):
    print(f"\nTop 8 by {col}:")
    print(blk.sort_values(col, ascending=False)[["block", "type", "margin_drift", "action_drift"]].head(8).to_string(index=False))
print("\nrank correlation (Spearman) alarm-aware vs action-aware:", round(blk["margin_drift"].rank().corr(blk["action_drift"].rank()), 2))
print("ranking uncertainty (mean bootstrap Jaccard):",
      {k: round(v["mean_jaccard"], 2) for k, v in (sens.get("ranking_uncertainty") or {}).items() if isinstance(v, dict)})


## 5b. Shadow check: does 4-bit alone move the monitor on identical inputs?
The fp16 robot runs 3 held-out test slots per task while a 4-bit shadow copy sees exactly the same camera frames. Any change in the monitor between the two is caused by quantization alone, not by a different trajectory. About 40 min; read the READING line before the long variant runs.

In [ ]:
collect(["--model", MODEL, "--variant", "fp16", "--shadow", "w4", "--suite", SUITE, "--condition", CONDITION,
         "--episodes", str(R_REF), "--ref_roles", "test", "--max_per_task", "3", "--out", str(WORK / "rollouts_shadow")])
run(["scripts/shadow_check.py", "--model", MODEL, "--key", KEY, "--shadow_root", str(WORK / "rollouts_shadow"),
     "--out", str(WORK / "shadow_check.json")], tail=2500)

## 6. Four compressed variants on the 175 test slots
W4 on the whole expert, and W4 with 15 % of the expert protected by three rules at matched memory: alarm-aware (`margin_drift`), action-aware (`action_drift`), random. All run the same paired test slots as the reference. **Resumable.**

In [ ]:
for v in VARIANTS:
    print("=" * 30, v)
    collect(["--model", MODEL, "--variant", v, "--suite", SUITE, "--condition", CONDITION,
         "--episodes", str(R_REF), "--roles", "test"], tail=600)
    rj = json.loads((CODE / f"rollouts/{MODEL}/{v}/{KEY}/run.json").read_text())
    print(f"{v}: success {rj['success_rate']:.3f}, protected {rj.get('protected_frac_actual', 0):.3f} of the expert,"
          f" blocks {[b.split('.')[-1] for b in rj.get('protected_blocks', [])]}")

## 7. Evaluation and decision

In [ ]:
run(["scripts/03_evaluate.py", "--model", MODEL, "--key", KEY, "--monitors", "mahalanobis#A,mahalanobis"], tail=1500)
tr = pd.read_csv(CODE / f"results/eval_trials/{MODEL}__{KEY}.csv")
tr = tr[(tr.alpha.round(3) == 0.10) & (tr.monitor == "mahalanobis#A")]
print(len(tr), "trial rows;", tr.variant.unique())

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def boot(df, stat, B=2000):
    # task-clustered bootstrap: resample tasks, keep all their trials together
    tasks = df.task.unique(); vals = []
    groups = {t: df[df.task == t] for t in tasks}
    for _ in range(B):
        s = pd.concat([groups[t] for t in rng.choice(tasks, len(tasks))])
        vals.append(stat(s))
    vals = np.array([v for v in vals if v == v])
    return stat(df), np.percentile(vals, 5), np.percentile(vals, 95)

rows = []
for v in VARIANTS:
    d = tr[tr.variant == v]
    js = d[(d.ref_success == 1) & (d.var_success == 1)]
    f = d[d.var_success == 0]
    est, lo, hi = boot(js, lambda s: (s.var_alarm - s.ref_alarm).mean())
    rows.append(dict(variant=v, success=d.var_success.mean(), fp16_success=d.ref_success.mean(),
                     joint_successes=len(js), FPR_fp16=js.ref_alarm.mean(), FPR_variant=js.var_alarm.mean(),
                     D_FPR=est, D_FPR_90lo=lo, D_FPR_90hi=hi,
                     TPR_variant=f.var_alarm.mean() if len(f) else np.nan, failures=len(f)))
summary = pd.DataFrame(rows).round(3)
print("Paired false-alarm change vs fp16 on jointly successful test trials (90% task-clustered interval)")
print(summary.to_string(index=False))

def contrast(a, b):
    m = tr[tr.variant == a].merge(tr[tr.variant == b], on=["task", "idx"], suffixes=("_a", "_b"))
    js = m[(m.var_success_a == 1) & (m.var_success_b == 1)]
    jf = m[(m.var_success_a == 0) & (m.var_success_b == 0)]
    fpr = boot(js, lambda s: (s.var_alarm_a - s.var_alarm_b).mean())
    succ = boot(m, lambda s: (s.var_success_a - s.var_success_b).mean())
    tpr = (jf.var_alarm_a - jf.var_alarm_b).mean() if len(jf) else np.nan
    return dict(pair=f"{a} - {b}", joint_successes=len(js), FPR_diff=fpr[0], FPR_90lo=fpr[1], FPR_90hi=fpr[2],
                success_diff=succ[0], joint_failures=len(jf), TPR_diff=tpr)

A, T, Rn = VARIANTS[1], VARIANTS[2], VARIANTS[3]
con = pd.DataFrame([contrast(A, T), contrast(A, Rn), contrast(T, Rn)]).round(3)
print("\nRule-vs-rule (negative FPR_diff = first rule has fewer false alarms)")
print(con.to_string(index=False))

In [ ]:
# Pre-stated decision table (planning thresholds for this gate, not significance tests)
w4 = summary.set_index("variant").loc["w4"]
c_at = con.iloc[0]
w4_moves = (w4.D_FPR >= 0.03) and (w4.D_FPR_90lo > 0)
method_better = (c_at.FPR_diff <= -0.02) and (c_at.FPR_90hi < 0.01) and (c_at.success_diff >= -0.05) \
                and not (c_at.TPR_diff == c_at.TPR_diff and c_at.TPR_diff < -0.10)
action_fixes = summary.set_index("variant").loc[T].D_FPR <= 0.01
print(f"W4 moves false alarms (D_FPR >= 0.03, 90% lo > 0): {w4_moves}  [D_FPR = {w4.D_FPR:+.3f}]")
print(f"alarm-aware beats action-aware (FPR -0.02 or better, success within 0.05, detection within 0.10): {method_better}")
if not w4_moves:
    verdict = "NO-GO for the method: the monitor is robust to expert W4 here. Write up as a robustness audit; consider W4 of the VLM too before giving up."
elif method_better:
    verdict = "GO: W4 degrades the monitor and the alarm-aware rule preserves it better than the action-aware rule. Proceed to Stage A."
elif action_fixes:
    verdict = "NO-GO for the method: action-aware protection already preserves the monitor. Report the degradation finding."
else:
    verdict = "INCONCLUSIVE: W4 degrades the monitor but the rules do not separate. Check the ranking uncertainty and layer types before deciding; consider more test slots."
print("\nDECISION:", verdict)

# Layer-type composition of each protected set (SmolVLA expert: even = self-attention, odd = cross-attention)
for v in VARIANTS[1:]:
    rj = json.loads((CODE / f"rollouts/{MODEL}/{v}/{KEY}/run.json").read_text())
    ids = [int(b.split(".")[-1]) for b in rj.get("protected_blocks", []) if b.split(".")[-1].isdigit()]
    print(f"{v:28s} whole blocks {sorted(ids)}  self-attn {sum(i % 2 == 0 for i in ids)} / cross-attn {sum(i % 2 for i in ids)}"
          f"  filler share {rj.get('filler_share', 0):.2f}")

In [ ]:
# Save a compact bundle to send back for analysis
out = WORK / "gonogo_summary"; out.mkdir(exist_ok=True)
summary.to_csv(out / "variants.csv", index=False); con.to_csv(out / "contrasts.csv", index=False)
(out / "decision.json").write_text(json.dumps({"condition": CONDITION, "budget_pct_of_expert": BUDGET_PCT,
    "n_action_steps": N_ACTION_STEPS, "verdict": verdict, "w4_moves": bool(w4_moves),
    "method_better": bool(method_better)}, indent=2))
shutil.copy(CODE / f"results/sensitivity/{MODEL}.json", out / "sensitivity.json")
shutil.copy(CODE / f"results/eval_trials/{MODEL}__{KEY}.csv", out / "eval_trials.csv")
for v in ["fp16"] + VARIANTS:
    shutil.copy(CODE / f"rollouts/{MODEL}/{v}/{KEY}/run.json", out / f"run_{v}.json")
shutil.make_archive(str(WORK / "gonogo_summary"), "zip", out)
print("Send this file:", WORK / "gonogo_summary.zip")
print("Download it from this version's Output tab: vla-gonogo/gonogo_summary.zip")